In [17]:
import numpy as np
import sys,os
import MAS_library as MASL
import pickle as pk
import h5py as h5
from ngp_funcs import NGP_xyz_prop 
import skimage.measure as skmeasure
import ast

# grid_sbox = int(ast.literal_eval(sys.argv[-1]))
grid_sbox = 16

def mat_reshape(mat, grid1, grid2):
    '''
    Reshape input mat into (grid1, grid1, grid1, grid2, grid2, grid2) shape
    '''
    if len(mat.shape) == 3:
        mat_rs = np.reshape(mat, (grid1, grid2, grid1, grid2, grid1, grid2))
    else:
        mat_rs = np.reshape(mat, (grid1, grid2, grid1, grid2, grid1, grid2, *mat.shape[3:]))

    mat_rs = np.moveaxis(mat_rs, 1, 2)
    mat_rs = np.moveaxis(mat_rs, 4, 3)
    mat_rs = np.moveaxis(mat_rs, 3, 2)
    return mat_rs


def get_padded_mat(Npart, n_pad, grid_sbox, grid):
    Npart_pad = np.pad(Npart, n_pad, 'wrap')

    Npart_pad1 = np.zeros((grid, grid, grid, grid_sbox + 2*n_pad, grid_sbox + 2*n_pad, grid_sbox + 2*n_pad))
    fac = (grid_sbox + 2*n_pad)//grid_sbox    
    Npart_pad1_reduce = np.zeros((grid, grid, grid, grid_sbox, grid_sbox, grid_sbox))    
    xstart, ystart, zstart = n_pad, n_pad, n_pad
    for jx in range(grid):
        for jy in range(grid):
            for jz in range(grid):
                Npart_pad1[jx, jy, jz] = Npart_pad[xstart + jx * grid_sbox - n_pad:xstart + (jx + 1) * grid_sbox + n_pad,
                                                        ystart + jy * grid_sbox - n_pad:ystart + (jy + 1) * grid_sbox + n_pad,
                                                        zstart + jz * grid_sbox - n_pad:zstart + (jz + 1) * grid_sbox + n_pad]

                Npart_pad1_reduce[jx, jy, jz] = skmeasure.block_reduce(Npart_pad1[jx, jy, jz], (fac, fac, fac), np.mean)
    return Npart_pad1_reduce, Npart_pad1

def process_LH_sim(isim_fid):
    import numpy as np
    # nrand_sel_box = 512
    # nrand_sel_box = 64
    norm_delta = 500
    norm_vel = 500
    BoxSize = 25.
    grid = 8
    nrand_sel_box = grid**3
    # grid_sbox = 32
    MAS_type = 'NGP'
    grid_tot = grid_sbox * grid
    snapnums = [90, 84, 78, 70, 60]
    # snapnums = [90, 78, 60]
    
    
    # import numpy as np
    # np.random.seed(0)
    # rand_sel = np.sort(np.random.randint(0, grid**3, nrand_sel_box)).astype(int)
    # rand_sel = (np.arange(grid**3)[:nrand_sel_box]).astype(int)
    if nrand_sel_box < grid**3:
        import numpy as np
        np.random.seed(0)
        ind_all = np.arange(grid**3)
        rand_sel = (np.random.permutation(ind_all)[:nrand_sel_box]).astype(int)
    else:
        rand_sel = np.arange(grid**3)
    # /work/nvme/bdne  
    sdir = f'/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512'
    savefname_dmo_fields = f'{sdir}/DMO_fields_isim_{isim_fid}.npy'
    
    
    for js, snapnum in enumerate(snapnums):
        root = '/work/hdd/bdne/spandey3/camels_tng/DMO/LH'
        snap_fname =  f'{root}/LH_{isim_fid}/snapshot_0{snapnum}.hdf5'
    
        np.random.seed(0)
        df = h5.File(snap_fname, 'r')                
        pos_m_truth = df['PartType1']['Coordinates'][()]/1000.
        vel_m_truth = df['PartType1']['Velocities'][()]
        ids_m_truth = np.arange(len(pos_m_truth))
    
        rho_bar = len(pos_m_truth)/(BoxSize**3)
        vol_vox = (BoxSize/grid_tot)**3
        N_bar_vox = rho_bar * vol_vox
    
        Npart = np.float32(np.zeros((grid_tot, grid_tot, grid_tot)))
        MASL.MA(np.float32(pos_m_truth), Npart, BoxSize, MAS_type, verbose=False)
    
        Npart /= N_bar_vox
        Npart /= norm_delta
    
        Npart_rs = mat_reshape(Npart, grid, grid_sbox)
    
        npad1 = grid_sbox
        Npart_pad1_rs, _ = get_padded_mat(Npart, int(npad1), grid_sbox, grid)
    
        npad2 = 2*grid_sbox
        Npart_pad2_rs, _ = get_padded_mat(Npart, int(npad2), grid_sbox, grid)
    
        vel_m_part = np.float32(np.zeros((grid_tot, grid_tot, grid_tot, 3)))
    
        Npart_cic = np.float32(np.zeros((grid_tot, grid_tot, grid_tot)))
        MASL.MA(np.float32(pos_m_truth), Npart_cic, BoxSize, 'CIC', verbose=False)
        for jc in range(3):
            mom_jc = np.float32(np.zeros((grid_tot, grid_tot, grid_tot)))
            MASL.MA(np.float32(pos_m_truth), mom_jc, BoxSize, 'CIC', verbose=False, W=vel_m_truth[:,jc].astype(np.float32))
            vel_m_jc = mom_jc/Npart_cic
            vel_m_jc[~np.isfinite(vel_m_jc)] = 0.0
            vel_m_part[..., jc] = vel_m_jc/norm_vel
    
        vel_m_part_rs = mat_reshape(vel_m_part, grid, grid_sbox)
    
        dmo_fields_all_snap = np.concat((Npart_rs[...,None], Npart_pad1_rs[...,None], Npart_pad2_rs[...,None], vel_m_part_rs), axis=-1)
    
        if js == 0:
            dmo_fields_all = dmo_fields_all_snap
        else:
            dmo_fields_all = np.concatenate((dmo_fields_all, dmo_fields_all_snap), axis=-1)
    
    dmo_fields_all_rs = dmo_fields_all.reshape((grid**3, *dmo_fields_all.shape[3:]))
    
    np.save(savefname_dmo_fields, dmo_fields_all_rs.astype(np.float32)[rand_sel, ...])

    if isim_fid == 0:
        savefname_meta = f'{sdir}/metadata_fields_grid_{grid_sbox}_isim_{isim_fid}_nrandsubsel_{nrand_sel_box}_MAS_{MAS_type}_nsnaps_{len(snapnums)}.pkl'
        meta_dict = {
                'snapnums': snapnums,
                'rand_sel': rand_sel,
                'norm_delta': norm_delta,
                'norm_vel': norm_vel,
                'grid_sbox':grid_sbox,
                'grid':grid,
                'MAS_type':MAS_type,
                'nrand_sel_box':nrand_sel_box        
                }
        pk.dump(meta_dict, open(savefname_meta, 'wb'))

    return






In [18]:
import multiprocessing as mp
# if __name__ == '__main__':
# n_sims = 1100
n_sims_offset = 0
n_sims = 1000
# n_sims = 10
# n_cores = mp.cpu_count()
n_cores = 5
print(n_cores)

# Create a pool of worker processes
pool = mp.Pool(processes=n_cores)

# Distribute the simulations across the available cores
sims_per_core = n_sims // n_cores
sim_ranges = [(n_sims_offset + i * sims_per_core, n_sims_offset + (i + 1) * sims_per_core) for i in range(n_cores)]

print(sims_per_core, sim_ranges)

# Handle any remaining simulations
remaining_sims = n_sims % n_cores
if remaining_sims > 0:
    sim_ranges[-1] = (sim_ranges[-1][0], sim_ranges[-1][1] + remaining_sims)

# Run save_cic_densities function for each simulation range in parallel
results = [pool.apply_async(process_LH_sim, args=(ji,)) for sim_range in sim_ranges for ji in range(*sim_range)]

# Wait for all tasks to complete
[result.get() for result in results]

# Close the pool and wait for tasks to finish
pool.close()
pool.join()






5
200 [(0, 200), (200, 400), (400, 600), (600, 800), (800, 1000)]


/tmp/ipykernel_3075983/3645651745.py:112: RuntimeWarning: invalid value encountered in divide
  vel_m_jc = mom_jc/Npart_cic
/tmp/ipykernel_3075983/3645651745.py:112: RuntimeWarning: invalid value encountered in divide
  vel_m_jc = mom_jc/Npart_cic
/tmp/ipykernel_3075983/3645651745.py:112: RuntimeWarning: invalid value encountered in divide
  vel_m_jc = mom_jc/Npart_cic
/tmp/ipykernel_3075983/3645651745.py:112: RuntimeWarning: invalid value encountered in divide
  vel_m_jc = mom_jc/Npart_cic
/tmp/ipykernel_3075983/3645651745.py:112: RuntimeWarning: invalid value encountered in divide
  vel_m_jc = mom_jc/Npart_cic


In [1]:
import numpy as np
import sys,os
import MAS_library as MASL
import pickle as pk
import h5py as h5
from ngp_funcs import NGP_xyz_prop 
import skimage.measure as skmeasure
import ast

# add_space_token = bool(ast.literal_eval(sys.argv[-1]))
add_space_token = False

def mat_reshape(mat, grid1, grid2):
    '''
    Reshape input mat into (grid1, grid1, grid1, grid2, grid2, grid2) shape
    '''
    if len(mat.shape) == 3:
        mat_rs = np.reshape(mat, (grid1, grid2, grid1, grid2, grid1, grid2))
    else:
        mat_rs = np.reshape(mat, (grid1, grid2, grid1, grid2, grid1, grid2, *mat.shape[3:]))

    mat_rs = np.moveaxis(mat_rs, 1, 2)
    mat_rs = np.moveaxis(mat_rs, 4, 3)
    mat_rs = np.moveaxis(mat_rs, 3, 2)
    return mat_rs


def get_padded_mat(Npart, n_pad, grid_sbox, grid):
    Npart_pad = np.pad(Npart, n_pad, 'wrap')

    Npart_pad1 = np.zeros((grid, grid, grid, grid_sbox + 2*n_pad, grid_sbox + 2*n_pad, grid_sbox + 2*n_pad))
    fac = (grid_sbox + 2*n_pad)//grid_sbox    
    Npart_pad1_reduce = np.zeros((grid, grid, grid, grid_sbox, grid_sbox, grid_sbox))    
    xstart, ystart, zstart = n_pad, n_pad, n_pad
    for jx in range(grid):
        for jy in range(grid):
            for jz in range(grid):
                Npart_pad1[jx, jy, jz] = Npart_pad[xstart + jx * grid_sbox - n_pad:xstart + (jx + 1) * grid_sbox + n_pad,
                                                        ystart + jy * grid_sbox - n_pad:ystart + (jy + 1) * grid_sbox + n_pad,
                                                        zstart + jz * grid_sbox - n_pad:zstart + (jz + 1) * grid_sbox + n_pad]

                Npart_pad1_reduce[jx, jy, jz] = skmeasure.block_reduce(Npart_pad1[jx, jy, jz], (fac, fac, fac), np.mean)
    return Npart_pad1_reduce, Npart_pad1




def process_LH_sim(isim_fid):
    import numpy as np
    import sys,os
    import MAS_library as MASL
    import pickle as pk
    import h5py as h5
    from ngp_funcs import NGP_xyz_prop 
    import skimage.measure as skmeasure

    # nrand_sel_box = 256
    nrand_sel_box = 512
    BoxSize = 25.
    grid = 8
    nvocab = 64
    # Mstar_cut = 8.75
    Mstar_cut = 9.5
    grid_sbox = nvocab
    grid_tot = grid_sbox * grid

    # prop_min = np.array([9.0, 26.5, 26.5, 26.5, -0.45])
    # prop_max = np.array([12.0, 30.5, 30.5, 30.5, 0.45])

    prop_min = np.array([Mstar_cut, 27.5, -0.5])
    prop_max = np.array([12.0, 31.0, 0.5])


    Npoints_max_per_subvol = 24
    #sort by Mstar token:
    ind_token_to_sort = 0
    # add_space_token = False
    # add_space_token = True

    dim_pos = 3
    dim_prop = len(prop_min)
    dim_tot = dim_pos + dim_prop

    start_token = nvocab + 1
    space_token = nvocab + 2
    pad_token = nvocab + 3
    end_token = nvocab + 4

    # import numpy as np
    # np.random.seed(0)
    # rand_sel = np.sort(np.random.randint(0, grid**3, nrand_sel_box)).astype(int)
    # rand_sel = (np.arange(grid**3)[:nrand_sel_box]).astype(int)
    if nrand_sel_box < grid**3:
        import numpy as np
        np.random.seed(0)
        ind_all = np.arange(grid**3)
        rand_sel = (np.random.permutation(ind_all)[:nrand_sel_box]).astype(int)
    else:
        rand_sel = np.arange(grid**3)
        
    # sdir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH'
    sdir = f'/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/gal_props_ns512_3prop'    
    snapnum = 90
    savefname_gals = f'{sdir}/galaxy_props_isim_{isim_fid}.npy'


    snap_dir_base = f'/work/hdd/bdne/spandey3/camels_tng/hydro/'
    group_catalog = f'{snap_dir_base}/LH/LH_{isim_fid}/groups_0{snapnum}.hdf5'
    photo_catalog = f'{snap_dir_base}/Photometry/IllustrisTNG/L25n256/LH/IllustrisTNG_LH_{isim_fid}_photometry.hdf5'
    # open the catalogue
    with h5.File(photo_catalog, "r") as hf:
        subhalo_index = np.array(hf[f"snap_0{snapnum}/SubhaloIndex"][:], dtype=int)
        g_band = np.log10(hf[f"snap_0{snapnum}/BC03/photometry/luminosity/attenuated/SLOAN/SDSS.g"][:])
        r_band = np.log10(hf[f"snap_0{snapnum}/BC03/photometry/luminosity/attenuated/SLOAN/SDSS.r"][:])
        i_band = np.log10(hf[f"snap_0{snapnum}/BC03/photometry/luminosity/attenuated/SLOAN/SDSS.i"][:])

    # Read the stellar masses of the subhalos/galaxies
    with h5.File(group_catalog, "r") as hf:
        M_star = np.log10(hf['Subhalo/SubhaloMassType'][:,4]*1e10 + 0.01) # Stellar masses in Msun/h
        pos = hf['Subhalo/SubhaloPos'][:]/1000.
        vel = hf['Subhalo/SubhaloVel'][:]/1000.

    M_star = M_star[subhalo_index]
    pos_h_truth = pos[subhalo_index]
    vel_h_truth = vel[subhalo_index][:,0]
    # prop_truth_all = np.stack((M_star, g_band, r_band, i_band, vel_h_truth)).T
    prop_truth_all = np.stack((M_star, r_band, vel_h_truth)).T
    # print(np.amin(vel_h_truth), np.amax(vel_h_truth))
    indsel = np.where(M_star > Mstar_cut)[0]
    prop_truth_all = prop_truth_all[indsel,:]
    pos_h_truth = pos_h_truth[indsel,:]


    dim_pos = pos_h_truth.shape[1]
    dim_prop = prop_truth_all.shape[1]


    Nhalos_truth = np.float32(np.zeros((grid_tot, grid_tot, grid_tot)))
    MASL.NGP(np.float32(pos_h_truth), Nhalos_truth, BoxSize)
    Nhalos_truth_rs = np.reshape(Nhalos_truth, (grid, grid_sbox, grid, grid_sbox, grid, grid_sbox))
    Nhalos_truth_rs = mat_reshape(Nhalos_truth, grid, grid_sbox)

    nMax_points = int(np.amax(Nhalos_truth_rs))

    dfhalo_ngp_wxyz_props = np.float32(np.zeros((grid_tot, grid_tot, grid_tot, nMax_points, dim_pos + dim_prop)))
    NGP_xyz_prop(np.float32(pos_h_truth), np.float32(prop_truth_all), dfhalo_ngp_wxyz_props, BoxSize)
    dfhalo_ngp_wxyz_props_rs = mat_reshape(dfhalo_ngp_wxyz_props, grid, grid_sbox)


    if add_space_token:
        max_sentence_length = 1 + Npoints_max_per_subvol*dim_tot + 1 + (Npoints_max_per_subvol - 1)
    else:
        max_sentence_length = 1 + Npoints_max_per_subvol*dim_tot + 1

    bins_digitize = np.linspace(-1e-3, 1, nvocab)
    Ntot_sel_final = 0
    sentences_all = np.zeros((grid, grid, grid, max_sentence_length), dtype=np.int16)
    for jx in range(grid):
        for jy in range(grid):
            for jz in range(grid):
                all_points_props_here = dfhalo_ngp_wxyz_props_rs[jx, jy, jz,...]
                Npoints_here = Nhalos_truth_rs[jx, jy, jz]
                indsel = np.where(Npoints_here > 0)
                Npoints_sel = Npoints_here[indsel]
                Npoints_sel_tot = int(np.sum(Npoints_sel))
                all_points_props_here_sel = all_points_props_here[indsel]
                word_array_all = []
                if len(Npoints_sel) > 0:    
                    for jc1 in range(len(Npoints_sel)):
                        Npoints_jc = (Npoints_sel[jc1]).astype(np.int16)
                        for jc2 in range(Npoints_jc):
                            all_points_props_here_sel_per_point = (all_points_props_here_sel[jc1, jc2, :])
                            position_token = np.array([indsel[0][jc1],indsel[1][jc1],indsel[2][jc1]], dtype=np.int16)
                            props_tokens = []
                            all_props = all_points_props_here_sel_per_point[3:]
                            for jp in range(len(all_props)):
                                prop_norm = np.clip((all_props[jp] - prop_min[jp]) / (prop_max[jp] - prop_min[jp]), 0.001, 0.999)
                                prop_token = np.digitize(prop_norm, bins_digitize).astype(np.int16)
                                if prop_token == 0:
                                    print(jp, prop_norm, prop_token)
                                props_tokens.append(prop_token)
                            props_token = np.array(props_tokens, dtype=np.int16)
                            # all_tokens = np.concatenate((position_token, props_tokens))
                            all_tokens = np.concatenate((props_tokens, position_token))
                            word_array_all.append(all_tokens)
                    word_array_all = np.array(word_array_all, dtype=np.int16)
                    tosort_token_all = word_array_all[:, ind_token_to_sort]
                    sort_inds = np.flip(np.argsort(tosort_token_all))
                    word_array_all = word_array_all[sort_inds]
                    if Npoints_sel_tot > Npoints_max_per_subvol:
                        # print(isim_fid, ' LH-SIM HAS MORE POINTS (',Npoints_sel_tot,Npoints_max_per_subvol, ') THAN MAXIMUM IN THE', jx, jy, jz, ' THIS SUBVOLUME!!! max-sent-length: ',max_sentence_length)
                        word_array_all = word_array_all[:Npoints_max_per_subvol]    
                    Ntot_sel_final += len(word_array_all)
                    if add_space_token:
                        space_array = (np.array(np.zeros(word_array_all.shape[0]) + space_token, dtype=np.int16))[:,None]
                        word_array_all_concat = np.concatenate((word_array_all, space_array), axis=1)
                        sentence_here = np.concatenate(([start_token],(word_array_all_concat).flatten()[:-1], [end_token]))
                    else:
                        sentence_here = np.concatenate(([start_token],(word_array_all).flatten(), [end_token]))
                else:
                    sentence_here = np.array([start_token, end_token], dtype=np.int16)

                npad = max_sentence_length - len(sentence_here)
                if npad > 0:
                    # pad_mat = np.array(np.zeros((npad, dim_tot)) + pad_token, dtype=np.int16)
                    # sentence_pad = pad_mat.flatten()
                    sentence_pad = np.array(np.zeros(npad) + pad_token, dtype=np.int16)
                    sentence_here = np.concatenate((sentence_here, sentence_pad))
                    
                sentences_all[jx, jy, jz] = sentence_here

    story_full = sentences_all.reshape((grid**3, max_sentence_length))
    if int(np.sum(Nhalos_truth)) > Ntot_sel_final:
        print(isim_fid, np.round(int(np.sum(Nhalos_truth))/Ntot_sel_final, 3))

    np.save(savefname_gals, story_full.astype(np.int16)[rand_sel, ...])

    if isim_fid == 0:
        savefname_meta = f'{sdir}/metadata_galaxy_props_snap_{snapnum}_grid_{grid_sbox}_isim_{isim_fid}_nrandsubsel_{nrand_sel_box}_nvocab{nvocab}_spacetoken_{add_space_token}_wSDSS_photometry_velx_Mstarcut_{Mstar_cut}.pkl'
        meta_dict = {
            'max_sentence_length': max_sentence_length,
            'grid': grid,
            'grid_sbox': grid_sbox,
            'Npoints_max_per_subvol': Npoints_max_per_subvol,
            'BoxSize': BoxSize,
            'prop_min': prop_min,
            'prop_max': prop_max,
            'nvocab':nvocab,
            'start_token': start_token,
            'pad_token': pad_token,
            'end_token': end_token,
            'space_token': space_token,
            'bins_digitize':bins_digitize,
            'rand_sel': rand_sel,
            'nrand_sel_box':nrand_sel_box,
            'add_space_token':add_space_token,
            'Mstar_cut':Mstar_cut
            }
        pk.dump(meta_dict, open(savefname_meta, 'wb'))

    return
    




In [2]:

import multiprocessing as mp
if __name__ == '__main__':
    # n_sims = 1100
    n_sims_offset = 0
    n_sims = 1000
    # n_cores = mp.cpu_count()
    n_cores = 10
    print(n_cores)

    # Create a pool of worker processes
    pool = mp.Pool(processes=n_cores)

    # Distribute the simulations across the available cores
    sims_per_core = n_sims // n_cores
    sim_ranges = [(n_sims_offset + i * sims_per_core, n_sims_offset + (i + 1) * sims_per_core) for i in range(n_cores)]

    print(sims_per_core, sim_ranges)

    # Handle any remaining simulations
    remaining_sims = n_sims % n_cores
    if remaining_sims > 0:
        sim_ranges[-1] = (sim_ranges[-1][0], sim_ranges[-1][1] + remaining_sims)

    # Run save_cic_densities function for each simulation range in parallel
    results = [pool.apply_async(process_LH_sim, args=(ji,)) for sim_range in sim_ranges for ji in range(*sim_range)]

    # Wait for all tasks to complete
    [result.get() for result in results]

    # Close the pool and wait for tasks to finish
    pool.close()
    pool.join()



10
100 [(0, 100), (100, 200), (200, 300), (300, 400), (400, 500), (500, 600), (600, 700), (700, 800), (800, 900), (900, 1000)]


10 1.039
8 1.062
13 1.028
24 1.013
26 1.032
35 1.089
32 1.061
28 1.016
36 1.069
48 1.095
46 1.05
47 1.034
53 1.02
60 1.037
59 1.002
66 1.06
67 1.05
69 1.003
65 1.031
70 1.05
72 1.101
76 1.015
81 1.041
77 1.037
92 1.008
118 1.055
113 1.011
129 1.037
133 1.026
140 1.007
142 1.003
153 1.019
162 1.017
163 1.007
175 1.019
174 1.149
181 1.007
191 1.011
188 1.011
194 1.016
131 1.101
198 1.039
190 1.012
203 1.06
210 1.081
216 1.046
219 1.063
215 1.069
221 1.016
217 1.188
231 1.116
236 1.087
238 1.226
229 1.006
234 1.008
235 1.01
246 1.023
245 1.048
255 1.05
258 1.016
259 1.098
260 1.011
267 1.013
272 1.009
287 1.032
288 1.053
289 1.016
291 1.109
300 1.008
299 1.079
302 1.004
305 1.08
311 1.012
309 1.054
308 1.035
313 1.012
322 1.088
321 1.084
328 1.058
327 1.026
331 1.006
337 1.018
333 1.03
339 1.005
340 1.013
348 1.027
349 1.013
351 1.008
355 1.019
353 1.002
359 1.035
360 1.004
364 1.022
285 1.042
382 1.012
396393  1.1121.132

402 1.029
401 1.068
399 1.081
403 1.003
413 1.024
419 1.114
422 1.

In [ ]:
import os
import shutil
import numpy as np
import torch

from nvidia.dali.pipeline import pipeline_def
import nvidia.dali.fn as fn
import nvidia.dali.types as types
from nvidia.dali.plugin.pytorch import DALIGenericIterator, LastBatchPolicy

class ExternalInputIterator:
    def __init__(self, input_dir, label_dir, batch_size, shuffle=True):
        self.batch_size = batch_size
        self.input_dir = input_dir
        self.label_dir = label_dir

        # Find all .npy files and sort them to ensure they match
        self.input_files = sorted([os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')])
        self.label_files = sorted([os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith('.npy')])

        if len(self.input_files) != len(self.label_files):
            raise ValueError(f"Mismatch in number of input and label files: "
                             f"{len(self.input_files)} vs {len(self.label_files)}")

        self.data_set_len = len(self.input_files)
        self.indices = np.arange(self.data_set_len)
        if shuffle:
            np.random.shuffle(self.indices)

    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.data_set_len:
            self.__iter__() # Reset for the next epoch
            raise StopIteration

        batch_inputs = []
        batch_labels = []

        # Gather a batch of data
        for _ in range(self.batch_size):
            if self.i >= self.data_set_len:
                # This handles the last batch if it's smaller than batch_size
                break
            
            idx = self.indices[self.i]
            
            # Load the numpy arrays from the files
            input_data = np.load(self.input_files[idx])
            label_data = np.load(self.label_files[idx])
            
            batch_inputs.append(input_data)
            batch_labels.append(label_data)
            
            self.i += 1

        return (batch_inputs, batch_labels)

    @property
    def size(self):
        return self.data_set_len


def create_dali_pipeline(input_dir, label_dir, batch_size, num_threads, device_id, shuffle=False):
    eii = ExternalInputIterator(input_dir, label_dir, batch_size, shuffle=shuffle)

    @pipeline_def(batch_size=batch_size, num_threads=num_threads, device_id=device_id)
    def npy_pipeline():
        inputs, labels = fn.external_source(source=eii, num_outputs=2)
        gpu_inputs = inputs.gpu()
        gpu_labels = labels.gpu()
        return gpu_inputs, gpu_labels

    pipe = npy_pipeline()
    pipe.build()
    return pipe, eii.size

DATA_PATH = 'dali_data'
BATCH_SIZE = 5
NUM_THREADS = 4
DEVICE_ID = 0 # GPU device ID

# Check for available GPU
if not torch.cuda.is_available():
    raise RuntimeError("This example requires a GPU.")

# --- Setup ---
# input_dir, label_dir = create_dummy_data(DATA_PATH, num_samples=120)
input_dir = '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512'
label_dir  = '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/gal_props_ns512_3prop'
# Create the DALI pipeline
pipeline, iterator_size = create_dali_pipeline(
    input_dir=input_dir,
    label_dir=label_dir,
    batch_size=BATCH_SIZE,
    num_threads=NUM_THREADS,
    device_id=DEVICE_ID,
    shuffle=False
)

# --- DALI PyTorch Iterator ---
# This wraps the DALI pipeline for seamless use in PyTorch.
# `output_map` maps the pipeline outputs to dictionary keys.
# `last_batch_policy` ensures the last, possibly smaller, batch is not dropped.
# dali_iterator = DALIGenericIterator(
#     [pipeline],
#     ['data', 'label'],
#     reader_name="NPY_READER",
#     last_batch_policy=LastBatchPolicy.PARTIAL,
#     auto_reset=True
# )
dali_iterator = DALIGenericIterator(
    [pipeline],
    ['data', 'label'],
    last_batch_policy=LastBatchPolicy.PARTIAL,
    auto_reset=True
)

print("\n--- Starting Training Loop Simulation ---")
num_epochs = 2
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    # The iterator is the dataloader
    for i, batch in enumerate(dali_iterator):
        # The output is a list of dictionaries, one for each GPU. We have one.
        data_batch = batch[0]
        
        # DALI returns data directly as GPU tensors
        inputs = data_batch['data']
        labels = data_batch['label']

        print(
            f"  Batch {i+1}: "
            f"Inputs shape={inputs.shape}, type={inputs.dtype}, device={inputs.device} | "
            f"Labels shape={labels.shape}, type={labels.dtype}, device={labels.device}"
        )

        # --- Your PyTorch model training would go here ---
        # model.train()
        # optimizer.zero_grad()
        # outputs = model(inputs)
        # loss = criterion(outputs, labels.squeeze(-1)) # Squeeze label if needed
        # loss.backward()
        # optimizer.step()
        # ------------------------------------------------

print("\n--- Simulation Finished ---")

# --- Cleanup ---
try:
    shutil.rmtree(DATA_PATH)
    print(f"Cleaned up dummy data directory: '{DATA_PATH}'")
except OSError as e:
    print(f"Error cleaning up directory {DATA_PATH}: {e.strerror}")




--- Starting Training Loop Simulation ---

Epoch 1/2
  Batch 1: Inputs shape=torch.Size([5, 512, 16, 16, 16, 30]), type=torch.float32, device=cuda:0 | Labels shape=torch.Size([5, 512, 146]), type=torch.int16, device=cuda:0
  Batch 2: Inputs shape=torch.Size([5, 512, 16, 16, 16, 30]), type=torch.float32, device=cuda:0 | Labels shape=torch.Size([5, 512, 146]), type=torch.int16, device=cuda:0

Epoch 2/2
  Batch 1: Inputs shape=torch.Size([5, 512, 16, 16, 16, 30]), type=torch.float32, device=cuda:0 | Labels shape=torch.Size([5, 512, 146]), type=torch.int16, device=cuda:0
  Batch 2: Inputs shape=torch.Size([5, 512, 16, 16, 16, 30]), type=torch.float32, device=cuda:0 | Labels shape=torch.Size([5, 512, 146]), type=torch.int16, device=cuda:0

--- Simulation Finished ---
Cleaned up dummy data directory: 'dali_data'


In [10]:
import os
import shutil
import numpy as np
import torch

from nvidia.dali.pipeline import pipeline_def
import nvidia.dali.fn as fn
import nvidia.dali.types as types
from nvidia.dali.plugin.pytorch import DALIGenericIterator, LastBatchPolicy

# --- 1. Helper Function to Create Dummy Data ---
# This function creates a dummy dataset of .npy files to make the example runnable.
# In your actual use case, you will already have these directories.
def create_dummy_data(data_path='dali_data', num_samples=100, sample_shape=(3, 64, 64)):
    """Creates dummy .npy files for inputs and labels."""
    print("--- Creating dummy data ---")
    input_dir = os.path.join(data_path, 'inputs')
    label_dir = os.path.join(data_path, 'labels')

    if os.path.exists(data_path):
        shutil.rmtree(data_path)

    os.makedirs(input_dir)
    os.makedirs(label_dir)

    for i in range(num_samples):
        # Create a sample input array
        input_data = np.random.rand(*sample_shape).astype(np.float32)
        # Create a corresponding label (e.g., an integer class label)
        label_data = np.array([i % 10], dtype=np.int32) # 10 classes

        input_filename = os.path.join(input_dir, f'input_{i:04d}.npy')
        label_filename = os.path.join(label_dir, f'label_{i:04d}.npy')

        np.save(input_filename, input_data)
        np.save(label_filename, label_data)

    print(f"Created {num_samples} input/label pairs in '{data_path}'")
    return input_dir, label_dir

# --- 2. The External Source Iterator ---
# DALI's ExternalSource operator needs a Python iterator that can feed it data.
# We create a class that finds all corresponding x and y .npy files
# and yields them in batches.
class ExternalInputIterator:
    def __init__(self, input_dir, label_dir, batch_size, shuffle=True):
        self.batch_size = batch_size
        self.input_dir = input_dir
        self.label_dir = label_dir

        # Find all .npy files and sort them to ensure they match
        self.input_files = sorted([os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')])
        self.label_files = sorted([os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith('.npy')])

        if len(self.input_files) != len(self.label_files):
            raise ValueError(f"Mismatch in number of input and label files: "
                             f"{len(self.input_files)} vs {len(self.label_files)}")

        self.data_set_len = len(self.input_files)
        self.indices = np.arange(self.data_set_len)
        if shuffle:
            np.random.shuffle(self.indices)

    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.data_set_len:
            self.__iter__() # Reset for the next epoch
            raise StopIteration

        batch_inputs = []
        batch_labels = []

        # Gather a batch of data
        for _ in range(self.batch_size):
            if self.i >= self.data_set_len:
                # This handles the last batch if it's smaller than batch_size
                break
            
            idx = self.indices[self.i]
            
            # Load the numpy arrays from the files
            input_data = np.load(self.input_files[idx])
            label_data = np.load(self.label_files[idx])
            
            batch_inputs.append(input_data)
            batch_labels.append(label_data)
            
            self.i += 1

        return (batch_inputs, batch_labels)

    @property
    def size(self):
        return self.data_set_len

# --- 3. The DALI Pipeline Definition ---
# This defines the graph of operations for DALI. We use the modern `@pipeline_def`
# decorator, which is cleaner than the class-based approach.
def create_dali_pipeline(input_dir, label_dir, batch_size, num_threads, device_id, shuffle=True):
    """
    Creates and builds the DALI pipeline.
    """
    # 1. Create the Python iterator
    eii = ExternalInputIterator(input_dir, label_dir, batch_size, shuffle=shuffle)

    # 2. Define the DALI pipeline using the iterator as a source
    @pipeline_def(batch_size=batch_size, num_threads=num_threads, device_id=device_id)
    def npy_pipeline():
        # When `name` is provided, fn.external_source returns a single, indexable
        # DataNode object, not a tuple. We must access the outputs via indexing.
        external_data = fn.external_source(source=eii, name="NPY_READER")
        inputs = external_data[0]
        labels = external_data[1]
        
        # Move data to the GPU. This is where DALI shines.
        # Any augmentations would happen here, on the GPU.
        # For example: inputs = fn.rotate(inputs.gpu(), angle=fn.random.uniform(range=(-10, 10)))
        gpu_inputs = inputs.gpu()
        gpu_labels = labels.gpu()
        
        return gpu_inputs, gpu_labels

    # 3. Build and return the pipeline and the iterator size
    pipe = npy_pipeline()
    pipe.build()
    return pipe, eii.size

# --- 4. Main Execution Block ---
if __name__ == '__main__':
    # --- Configuration ---
    DATA_PATH = 'dali_data'
    BATCH_SIZE = 16
    NUM_THREADS = 4
    DEVICE_ID = 0 # GPU device ID

    # Check for available GPU
    if not torch.cuda.is_available():
        raise RuntimeError("This example requires a GPU.")

    # --- Setup ---
    input_dir, label_dir = create_dummy_data(DATA_PATH, num_samples=120)
    
    # Create the DALI pipeline
    pipeline, iterator_size = create_dali_pipeline(
        input_dir=input_dir,
        label_dir=label_dir,
        batch_size=BATCH_SIZE,
        num_threads=NUM_THREADS,
        device_id=DEVICE_ID,
        shuffle=True
    )

    # --- DALI PyTorch Iterator ---
    # This wraps the DALI pipeline for seamless use in PyTorch.
    # Because fn.external_source does not expose metadata, we must manually
    # provide the size of the dataset and remove the `reader_name` argument.
    dali_iterator = DALIGenericIterator(
        [pipeline],
        ['data', 'label'],
        size=iterator_size,
        last_batch_policy=LastBatchPolicy.PARTIAL,
        auto_reset=True
    )

    print("\n--- Starting Training Loop Simulation ---")
    num_epochs = 2
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        # The iterator is the dataloader
        for i, batch in enumerate(dali_iterator):
            # The output is a list of dictionaries, one for each GPU. We have one.
            data_batch = batch[0]
            
            # DALI returns data directly as GPU tensors
            inputs = data_batch['data']
            labels = data_batch['label']

            print(
                f"  Batch {i+1}: "
                f"Inputs shape={inputs.shape}, type={inputs.dtype}, device={inputs.device} | "
                f"Labels shape={labels.shape}, type={labels.dtype}, device={labels.device}"
            )

            # --- Your PyTorch model training would go here ---
            # model.train()
            # optimizer.zero_grad()
            # outputs = model(inputs)
            # loss = criterion(outputs, labels.squeeze(-1)) # Squeeze label if needed
            # loss.backward()
            # optimizer.step()
            # ------------------------------------------------

    print("\n--- Simulation Finished ---")

    # --- Cleanup ---
    try:
        shutil.rmtree(DATA_PATH)
        print(f"Cleaned up dummy data directory: '{DATA_PATH}'")
    except OSError as e:
        print(f"Error cleaning up directory {DATA_PATH}: {e.strerror}")




--- Creating dummy data ---


Created 120 input/label pairs in 'dali_data'


/u/spandey3/gotham2/lib/python3.12/site-packages/nvidia/dali/plugin/base_iterator.py:208: Warning: Please set `reader_name` and don't set last_batch_padded and size manually whenever possible. This may lead, in some situations, to missing some samples or returning duplicated ones. Check the Sharding section of the documentation for more details.
  _iterator_deprecation_warning()


RuntimeError: Don't know how to extract the shape out of <class 'list'>

In [2]:
import sys, os
input_dir = '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512'
dm_fields_files = sorted([os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')])




In [4]:
input_dir = '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512'
files_in_dir = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')]

# Define the key function to extract the integer
def get_sim_number(filepath):
    # DMO_fields_isim_123.npy -> 123
    filename = os.path.basename(filepath)
    number_part = filename.split('_')[-1] # Gets '123.npy'
    number = int(number_part.split('.')[0]) # Gets '123' and converts to integer
    return number

dm_fields_files = sorted(files_in_dir, key=get_sim_number)


In [5]:
# dm_fields_files


['/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_0.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_1.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_2.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_3.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_4.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_5.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_6.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_7.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_8.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fields_ns512/DMO_fields_isim_9.npy',
 '/work/nvme/bdne/spandey3/camels_tng/gotham_data/LH/DMO_fie